# Day 18 Revision Summary — Feature Engineering

- **Feature engineering is the biggest lever you have.** A feature is one input column; feature engineering means creating useful new columns, encoding text into numbers, and cleaning/scaling what's already there. A model can't invent information you didn't give it, but a good feature can hand it the answer directly.
- **The star demo: 0.70 → 0.99 with one column.** On a "healthy if x²+y² < 4" dataset, a logistic model on raw `[x, y]` scores ~0.695 (a straight line can't draw a circle). Adding one engineered column, `dist2 = x**2 + y**2`, lets the exact same model hit ~0.995 — one threshold on `dist2` splits the data cleanly.
- **One-hot encode categorical text, never label-encode it.** A column like `city = "KTM"` is text a model can't do math on. One-hot makes one 0/1 column per category (`is_KTM`, `is_PKR`, ...); mapping categories to 1, 2, 3 invents a fake ordering the model would wrongly learn from.
- **Scale numerics, impute gaps — inside a Pipeline.** Distance-based models (logistic, KNN, k-means) need `StandardScaler`; trees and forests don't. Fill numeric gaps with the median and categorical gaps with the most frequent value, and fit all of this on the *training* data only (inside a `Pipeline`) so no test information leaks in.
- **`ColumnTransformer` preps every column type in one object.** It routes numeric columns to `StandardScaler` and categorical columns to `OneHotEncoder(handle_unknown="ignore")` in a single step, wrapped in a `Pipeline` with the model — no manual steps to forget, and it evaluates honestly with `cross_val_score`.
- **Engineer generously, then select ruthlessly.** Useless columns add noise, hurt accuracy, and slow training down. `SelectKBest` (or a trained model's `feature_importances_`) tells you which columns actually earn their place.


## Homework (Day 18 — `homework/homework.md`, your 2-hour self-study)

Do these before Day 19.


## Assignment 1 — Engineer a feature that helps

**What's being asked:** Pick any dataset (e.g. `load_diabetes`, or your own CSV). Create one **ratio** or **difference** feature you think matters. Prove it helps by comparing `cross_val_score` with and without it. Write one sentence on why it did (or didn't) help.

**Approach:**
1. Load a small dataset with at least two numeric columns and a label (`load_diabetes`/`load_wine`, or the class's own `x, y, healthy` circle data both work).
2. Build a baseline pipeline (`StandardScaler` + `LogisticRegression`, or a regressor if the label is continuous) and score it on the *raw* columns with `cross_val_score(..., cv=5).mean()`.
3. Engineer one new column — a ratio (`col_a / col_b`) or a difference (`col_a - col_b`) — using `DataFrame.assign`.
4. Re-run the exact same pipeline and `cross_val_score` on raw + the new column.
5. Compare the two mean scores and write one sentence explaining why the new column helped (or didn't).


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# TODO 1: Build or load a small dataset with a label and at least two numeric
# columns. Reusing the class's own circle dataset is fine:
# rng = np.random.default_rng(42)
# x = rng.uniform(-3, 3, 600)
# y = rng.uniform(-3, 3, 600)
# healthy = ((x**2 + y**2) < 4).astype(int)
# df = pd.DataFrame({"x": x, "y": y})

# TODO 2: Baseline pipeline + score on the RAW columns only.
# model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
# raw = df[["x", "y"]]
# raw_score = cross_val_score(model, raw, healthy, cv=5).mean()
# print("raw:", raw_score)

# TODO 3: Engineer ONE ratio or difference column with df.assign(...).
# eng = raw.assign(new_col=...)

# TODO 4: Score the SAME pipeline on raw + the new column.
# eng_score = cross_val_score(model, eng, healthy, cv=5).mean()
# print("engineered:", eng_score)

# TODO 5: One sentence — why did (or didn't) the new column help?


## Assignment 2 — Encode two ways, confirm they match

**What's being asked:** Take a categorical column and one-hot it with both `pd.get_dummies` and `OneHotEncoder`. Confirm the 0/1 columns are the same. Why does `OneHotEncoder` also need `handle_unknown="ignore"` for production?

**Approach:**
1. Build a small `DataFrame` with one text column (the class's `city = ["KTM", "PKR", "BRT", "KTM"]` example, or your own categories).
2. Encode it with `pd.get_dummies(df["col"], prefix="col")`.
3. Encode it again with `OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit_transform(df[["col"]])`, and check `.categories_` for the column order sklearn picked.
4. Sort both results' columns into the same order and compare the underlying values (e.g. `np.array_equal` after aligning columns) to confirm they match.
5. Answer: `handle_unknown="ignore"` matters because a production model will eventually see a category it never saw during training (e.g. a new city) — without it, `OneHotEncoder.transform` raises an error on that unseen value; with it, the unseen category just becomes an all-zeros row instead of crashing prediction.


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

# TODO 1: A tiny table with one text column, "city".
df = pd.DataFrame({"city": ["KTM", "PKR", "BRT", "KTM"]})

# TODO 2: Quick way — pd.get_dummies.
dummies = pd.get_dummies(df["city"], prefix="city")
print(dummies)

# TODO 3: Production way — OneHotEncoder.
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded = enc.fit_transform(df[["city"]])
print(enc.categories_)
print(encoded)

# TODO 4: Align column order and confirm the two encodings match
# (dummies columns are alphabetical by category, same as enc.categories_ here).
# dummies_sorted = dummies.sort_index(axis=1)
# print(np.array_equal(dummies_sorted.to_numpy().astype(int), encoded.astype(int)))


   city_BRT  city_KTM  city_PKR
0     False      True     False
1     False     False      True
2      True     False     False
3     False      True     False
[array(['BRT', 'KTM', 'PKR'], dtype=object)]
[[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]


## Assignment 3 — Date parts

**What's being asked:** Given a column of dates, extract useful parts (day of week, month, day name) with pandas' `.dt` accessor, then explain why `dayofweek` is a better feature than the raw date string.

**Approach:**
1. Parse a `Series` of date strings with `pd.to_datetime`.
2. Pull out `.dt.dayofweek` (0 = Monday), `.dt.month`, and `.dt.day_name()`.
3. Answer: a raw date string like `"2026-07-27"` is effectively a unique category the model has almost never seen before (every date is different), so it can't generalize from it. `dayofweek` collapses that into 7 recurring buckets a model *can* learn a pattern from (e.g. weekend vs. weekday spending), the same idea as the BINS toolbox move from class (age → child/adult/senior).


In [ ]:
import pandas as pd

s = pd.to_datetime(pd.Series(["2026-07-27", "2026-12-25", "2026-01-01"]))
print(s.dt.dayofweek)   # 0 = Monday
print(s.dt.month)
print(s.dt.day_name())


## Assignment 4 — Predict then run

**What's being asked:** Before running the cell below, predict whether adding 20 columns of pure random noise will help, hurt, or do nothing to a logistic regression model's cross-validated accuracy on `load_breast_cancer` — then run it and check.

**Prediction (write yours before running):** Random noise columns carry no real signal, so in principle they should do nothing to the *Bayes-optimal* score — but in practice, with a finite dataset, a model can spuriously "fit" some of that noise (especially with regularized linear models it's usually small movement, but it also slows training and adds columns that mean nothing, which is a symptom `SelectKBest` would guard against on a less-regularized model).

**Approach:**
1. Run the baseline pipeline on the real `load_breast_cancer` features and record the mean `cross_val_score`.
2. Concatenate 20 columns of `rng.normal` noise onto the real features.
3. Re-run the same pipeline and compare the two scores.
4. Confirm the lesson: more columns is not automatically better — hence feature selection.


In [ ]:
import numpy as np, pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
print("real features   :", round(cross_val_score(model, X, y, cv=5).mean(), 3))

rng = np.random.default_rng(0)
junk = pd.DataFrame(rng.normal(size=(len(X), 20)), columns=[f"junk{i}" for i in range(20)])
X2 = pd.concat([X.reset_index(drop=True), junk], axis=1)
print("+ 20 junk cols  :", round(cross_val_score(model, X2, y, cv=5).mean(), 3))
# Lesson: more columns is not automatically better - hence feature selection.


## Coming up — Day 19

**Week 4 revision** — a full end-to-end project on mixed data: engineer features, build a `ColumnTransformer` pipeline, bake-off a few models, and pick a winner with cross-validation.
